# Sherkala-8B — проверка на RunPod A40

`inceptionai/Llama-3.1-Sherkala-8B-Chat` — стандартная архитектура Llama 3.1 (без custom code, в отличие от Qolda),
дообученная на казахском. Заявленные языки в карточке — **kk, en** (русский не заявлен явно, в отличие от Qolda — проверяем отдельно).

Зрения у модели нет — это чисто текстовое сравнение качества kk/ru/en с Qolda ([01_qolda_check.ipynb](01_qolda_check.ipynb)) и, позже, с KazLLM-8B.

Лицензия репозитория — `cc-by-nc-sa-4.0` (некоммерческая) — учесть при решении про продакшн, сейчас не блокирует тест.

Запускать на том же поде, что и Qolda-ноутбук. Если Qolda ещё держит модель в VRAM — либо перезапусти ядро того ноутбука, либо освободи память (`del model; torch.cuda.empty_cache()`), 8B в bf16 занимает ~16 GB.

In [ ]:
# --- 0. Проверка GPU ---
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
    print('vram total GB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
    print('vram free GB:', round((torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 2**30, 1))
else:
    raise SystemExit('GPU недоступна в этом контейнере.')

In [ ]:
# --- HF-токен для gated-репозитория ---
# Sherkala и KazLLM-8B оба "gated: auto" — доступ выдаётся сразу после того, как на странице
# модели на huggingface.co нажать "Request access" (форма), но нужен токен, чтобы это подтвердить
# программно. Создать: huggingface.co/settings/tokens -> New token -> тип Read.
#
# НЕ коммить этот токен в git. Вставь строку сюда, выполни ячейку один раз за сессию пода
# (значение останется в памяти ядра до перезапуска) и по желанию сотри саму ячейку.
import os

if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = 'hf_ВСТАВЬ_СВОЙ_ТОКЕН_СЮДА'

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])
print('HF login OK')

In [ ]:
# --- 1. Кэш в /workspace (переживает рестарт пода, / — всего 5GB) ---
import os
os.environ['HF_HOME'] = '/workspace/hf_cache'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.makedirs('/workspace/hf_cache', exist_ok=True)
print('HF_HOME =', os.environ['HF_HOME'])

In [ ]:
# --- 2. Зависимости ---
# Стандартная Llama-архитектура, без custom code — версия transformers не так критична,
# как для Qolda. Но не ставим -U вслепую — берём известную рабочую линейку.
%pip install -q "transformers==4.55.0" accelerate sentencepiece hf_transfer

In [ ]:
import transformers
print('transformers version:', transformers.__version__)

## Загрузка модели

`AutoModelForCausalLM` + `AutoTokenizer`, `trust_remote_code` не требуется — это ключевое отличие от Qolda,
где кастомная `InternVLChatModel` требовала точной версии transformers. Здесь риск несовместимости версий ниже.

In [ ]:
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'inceptionai/Llama-3.1-Sherkala-8B-Chat'

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).eval().cuda()
print(f'Загружено за {time.time()-t0:.1f}s')
print('VRAM занято, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

In [ ]:
# Подстраховка: если чат-темплейт не пришёл вместе с токенайзером (бывает на некоторых
# зеркалах/версиях), назначаем его руками — темплейт из карточки модели (Llama 3.1 chat format).
FALLBACK_CHAT_TEMPLATE = (
    "{% set loop_messages = messages %}"
    "{% for message in loop_messages %}"
    "{% set content = '<|start_header_id|>' + message['role'] + '<|end_header_id|>\n\n' "
    "+ message['content'] | trim + '<|eot_id|>' %}"
    "{% if loop.index0 == 0 %}{% set content = bos_token + content %}{% endif %}"
    "{{ content }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|start_header_id|>assistant<|end_header_id|>\n\n' }}"
    "{% endif %}"
)

if tokenizer.chat_template is None:
    print('chat_template отсутствует в токенайзере — назначаю fallback из карточки модели.')
    tokenizer.chat_template = FALLBACK_CHAT_TEMPLATE
else:
    print('chat_template уже встроен в токенайзер — использую его как есть.')

## Тест 1 — языки (kk/ru/en)

Те же промпты, что и в `01_qolda_check.ipynb`, для прямого сравнения бок о бок.

In [ ]:
GEN_KW = dict(max_new_tokens=512, do_sample=False, pad_token_id=tokenizer.eos_token_id)

def ask_text(question):
    messages = [{'role': 'user', 'content': question}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt'
    ).cuda()
    with torch.no_grad():
        out = model.generate(input_ids, **GEN_KW)
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

prompts = {
    'kk': 'Қазақстанның бас қаласы қандай және неге астана болып таңдалды? Қысқаша жауап бер.',
    'ru': 'Объясни своими словами, чем отличается ИП от ТОО в Казахстане. Кратко.',
    'en': 'In two sentences, explain what a vector database is used for in RAG systems.',
}

for lang, q in prompts.items():
    t0 = time.time()
    answer = ask_text(q)
    dt = time.time() - t0
    print(f'=== {lang} ({dt:.1f}s) ===')
    print('Q:', q)
    print('A:', answer)
    print()

**На что смотреть, помимо общего качества:** карточка модели явно заявляет только kk/en — русский под вопросом. Смотрим, не съезжает ли ответ на `ru` в казахский/английский или не деградирует ли грамматика заметнее, чем у Qolda.

## Тест 2 — tool calling

Не входит в scope этого этапа по решению пользователя, но раз база — Llama 3.1 (в отличие от Qolda, где tool calling не заявлен вообще),
штатный формат стоит проверить одним быстрым прогоном — пригодится на этапе 3.

In [ ]:
import json

tools = [{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': 'Получить текущую погоду в городе',
        'parameters': {
            'type': 'object',
            'properties': {'city': {'type': 'string', 'description': 'Название города'}},
            'required': ['city'],
        },
    },
}]

messages = [{'role': 'user', 'content': 'Қандай ауа райы Алматыда қазір?'}]

try:
    input_ids = tokenizer.apply_chat_template(
        messages, tools=tools, add_generation_prompt=True, return_tensors='pt'
    ).cuda()
    with torch.no_grad():
        out = model.generate(input_ids, **GEN_KW)
    new_tokens = out[0][input_ids.shape[-1]:]
    tool_answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    print(tool_answer)
except Exception as e:
    print('Chat template не принимает tools= напрямую (ожидаемо, если темплейт был заменён на fallback):', e)
    print('Это не финальный вывод по tool calling — нужен отдельный целевой прогон на этапе 3.')

**Как читать результат:** если модель вернула что-то похожее на вызов функции (JSON с `name`/`arguments`, либо специальный тег) — формат жив и наследуется от Llama 3.1 после дообучения. Если ответила обычным текстом — либо дообучение это стёрло, либо нужен другой промпт/системный формат. Это не финальная проверка, просто быстрый сигнал.

## Замер VRAM

In [ ]:
print('VRAM allocated, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))
print('VRAM reserved,  GB:', round(torch.cuda.memory_reserved() / 2**30, 2))
print('VRAM peak,      GB:', round(torch.cuda.max_memory_allocated() / 2**30, 2))

## Освободить память перед следующей моделью

In [ ]:
# Запусти перед тем, как переключаться на KazLLM-8B в другом ноутбуке на этом же поде.
del model
torch.cuda.empty_cache()
print('VRAM после очистки, GB:', round(torch.cuda.memory_allocated() / 2**30, 2))

## Дальше по плану

1. Сравнить ответы на kk/ru/en с `01_qolda_check.ipynb` бок о бок — записать наблюдения в `docs/findings.md`.
2. Тот же прогон для `issai/LLama-3.1-KazLLM-1.0-8B` (`03_kazllm_check.ipynb`).
3. Если tool calling выглядит рабочим — отдельный целевой ноутбук на этапе 3, с реальными тулами проекта.